# 测试提示词优化功能

测试 GPT API 是否正常工作

In [1]:
import os
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == 'tests' else Path.cwd()
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

Project root: e:\Study\Tsinghua_Project\KINYO_Project


## 1. 检查环境变量

In [2]:
# 检查 API Key
k_token_key = os.getenv("K_TOKEN_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")
k_token_base = os.getenv("K_TOKEN_BASE_URL", "https://ai.ktokenhub.app")
generation_model = os.getenv("SCRIPT_MODEL", "gpt-5.4")

print(f"K_TOKEN_API_KEY: {'已设置' if k_token_key else '未设置'}")
print(f"OPENAI_API_KEY: {'已设置' if openai_key else '未设置'}")
print(f"使用的 API Key: {k_token_key or openai_key or '无'}")
print(f"K_TOKEN_BASE_URL: {k_token_base}")
print(f"GENERATION_MODEL: {generation_model}")

api_key = k_token_key or openai_key
if not api_key:
    print("\n⚠️ 警告: 没有设置 API Key!")
    print("请设置环境变量: K_TOKEN_API_KEY 或 OPENAI_API_KEY")

K_TOKEN_API_KEY: 未设置
OPENAI_API_KEY: 已设置
使用的 API Key: sk-b1a560f75e9a7130fea840ad9f4c5481ab5a5a1c658f4ad62ad56302cfa46887
K_TOKEN_BASE_URL: https://ai.ktokenhub.app
GENERATION_MODEL: gpt-5.4


## 2. 测试 OpenAI 客户端连接

In [3]:
from openai import OpenAI

api_key = os.getenv("K_TOKEN_API_KEY") or os.getenv("OPENAI_API_KEY")
base_url = os.getenv("K_TOKEN_BASE_URL", "https://ai.ktokenhub.app/v1")
model = os.getenv("SCRIPT_MODEL", "gpt-5.4")

if api_key:
    client = OpenAI(api_key=api_key, base_url=base_url)
    
    # 简单测试
    test_prompt = "请用一句话回复：你好"
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": test_prompt}
            ],
            temperature=0.7
        )
        
        print(f"✅ API 连接成功!")
        print(f"模型: {model}")
        print(f"响应: {response.choices[0].message.content}")
        
    except Exception as e:
        print(f"❌ API 调用失败: {e}")
        print(f"错误类型: {type(e).__name__}")
else:
    print("❌ 没有设置 API Key，无法测试")

✅ API 连接成功!
模型: gpt-5.4
响应: 你好


## 3. 测试提示词优化函数

In [4]:
from video_agent.agent.video_segment_generator import (
    optimize_prompt_with_gpt,
    build_image_index_map,
    load_seedance_guide
)

# 测试数据
test_segment = {
    "stage": "S01",
    "visual": "茶几俯拍摆开主机、高清线、麦克风",
    "voiceover": "K7家庭K歌机，让家秒变KTV",
    "camera": "中景",
    "motion": "缓推"
}

test_characters = [
    {
        "id": "Alice",
        "description": "年轻女性，长发，穿白色连衣裙",
        "selected_images": ["front_full"]
    },
    {
        "id": "小明",
        "description": "中年男性，戴眼镜，穿蓝色衬衫",
        "selected_images": ["front_masked"]
    }
]

test_initial_prompt = """参考图片1作为场景参考

镜头1：中景，产品 茶几俯拍摆开主机、高清线、麦克风。

全程画面高清电影纪实风，色调温暖，光影柔和；
保持无字幕，不要生成Logo，不要生成水印。
产品K7家庭K歌主机的外观保持不变，产品logo清晰可见。
"""

# 构建图片索引映射
image_index_map = build_image_index_map(
    has_keyframe=True,
    has_product=True,
    character_infos=test_characters
)

print("图片索引映射:")
for k, v in sorted(image_index_map.items(), key=lambda x: x[1]):
    print(f"  图片{v} = {k}")

图片索引映射:
  图片1 = keyframe
  图片2 = product
  图片3 = Alice_front_full
  图片4 = 小明_front_masked


In [5]:
# 调用优化函数
print("开始优化提示词...")
print(f"初始提示词长度: {len(test_initial_prompt)} 字符\n")

optimized = optimize_prompt_with_gpt(
    segment=test_segment,
    character_infos=test_characters,
    initial_prompt=test_initial_prompt,
    duration=5,
    product_image_name="K7主机",
    image_index_map=image_index_map
)

print(f"优化后提示词长度: {len(optimized)} 字符\n")
print("="*60)
print("优化后的提示词:")
print("="*60)
print(optimized)

开始优化提示词...
初始提示词长度: 118 字符

[DEBUG] Calling GPT API: model=gpt-5.4, base_url=https://ai.ktokenhub.app/v1
[DEBUG] API Key set: True
[DEBUG] Standard OpenAI response, length: 404 chars
优化后提示词长度: 404 字符

优化后的提示词:
参考图片1中的场景环境与构图氛围，将图片2中的K7家庭K歌主机定义为产品K7，将图片3中长发、穿白色连衣裙的年轻女性定义为Alice，将图片4中戴眼镜、穿蓝色衬衫的中年男性定义为小明。

镜头1：中景，缓慢推镜，Alice站在茶几一侧，身体微微前倾，右手轻轻指向摆放在茶几中央的产品K7，左手自然垂落；小明站在茶几另一侧，头部微微点向产品，右手轻扶茶几边缘，动作自然连贯。茶几上清晰摆开产品K7、高清线、麦克风，产品位于画面中心，位置稳定，外观保持与图片2一致，产品logo清晰可见。场景保持图片1中的空间布局与温暖生活化氛围，室内光影柔和，色调温暖真实。音频为自然、清晰、无机械感的中文男声旁白，语速适中，声音有情感起伏，台词：{K7家庭K歌机，让家秒变KTV}。

全程高清，细节丰富，电影纪实风，画面自然稳定，人物面部稳定不变形，动作流畅无卡顿，保持无字幕、无Logo（产品自身logo除外）、无水印。


In [6]:
response

'<!doctype html>\n<html lang="zh-CN">\n  <head>\n    <meta charset="UTF-8" />\n    <link rel="icon" type="image/png" href="/logo.png" />\n    <meta name="viewport" content="width=device-width, initial-scale=1.0" />\n    <title>Ktokenhub - AI API Gateway</title>\n    <script type="module" crossorigin src="/assets/index-BNpu0I2z.js"></script>\n    <link rel="modulepreload" crossorigin href="/assets/vendor-vue-CUflzYHr.js">\n    <link rel="modulepreload" crossorigin href="/assets/vendor-i18n-Crpujnnm.js">\n    <link rel="modulepreload" crossorigin href="/assets/vendor-misc-B3tlMzbo.js">\n    <link rel="stylesheet" crossorigin href="/assets/index-DNBplF6D.css">\n  <script nonce="RTrwE3wOOtqSFuR53HEvtw==">window.__APP_CONFIG__={"registration_enabled":false,"email_verify_enabled":false,"registration_email_suffix_whitelist":[],"promo_code_enabled":true,"password_reset_enabled":false,"invitation_code_enabled":false,"totp_enabled":false,"turnstile_enabled":false,"turnstile_site_key":"","site_na

## 4. 对比优化前后

In [ ]:
print("="*60)
print("【优化前】")
print("="*60)
print(test_initial_prompt)
print()
print("="*60)
print("【优化后】")
print("="*60)
print(optimized)
print()

if test_initial_prompt.strip() == optimized.strip():
    print("⚠️ 警告: 优化前后提示词完全相同!")
else:
    print("✅ 提示词已被优化")

## 5. 检查 Seedance 指南是否加载成功

In [ ]:
guide = load_seedance_guide()
print(f"Seedance 指南加载状态: {'成功' if guide else '失败'}")
print(f"指南长度: {len(guide)} 字符")
if guide:
    print(f"\n指南前500字符:\n{guide[:500]}...")
else:
    print("\n⚠️ 指南未加载，请检查文件路径:")
    from video_agent.agent.video_segment_generator import SEEDANCE_GUIDE_PATH
    print(f"  预期路径: {SEEDANCE_GUIDE_PATH}")
    print(f"  文件存在: {SEEDANCE_GUIDE_PATH.exists()}")

## 6. 直接打印发送给 GPT 的完整 prompt

In [ ]:
# 手动构建 prompt 看看
seedance_guide = load_seedance_guide()

system_prompt = f"""你是视频生成提示词专家，专门优化 Doubao Seedance 2.0 的提示词。

## Seedance 提示词指南参考

{seedance_guide[:5000]}

## 核心优化要求

### 1. 图片引用格式（关键！）
- 必须使用"图片1、图片2..."指代素材，禁止用模糊表述
- 图片编号规则：图片1=关键帧，图片2=产品图（如有），图片3开始=人物图
- 定义主体时必须标明对应图片：将图片N中的[特征]定义为[人物名]
- 引用主体时保持名称一致

### 2. 结构要求
- 必须使用"镜头1、镜头2"分镜时序描述
- 每个镜头包含：运镜 + 动作 + 位置 + 音频
- 时长较短（<=5秒）时动作要简洁，不宜过多镜头切换

### 3. 动作描述
- 具体到肢体部位（手、腿、头、肩）
- 优先低缓连续小动作
- 情绪外化为具体动作细节（如"嘴角上扬"代替"开心"）

### 4. 音频约束
- 人物说话使用自然、清晰的音色
- 避免机械感、AI口音
- 语速适中，声音有情感起伏
- 台词用 {{中文}} 格式

### 5. 画质约束
- 保持无字幕、无Logo、无水印
- 人物面部稳定不变形
- 动作流畅无卡顿

### 6. 多人物场景
- 明确标注每个人物对应哪张图片
- 人物名称前后一致
- 避免人物混淆或重复

请直接输出优化后的提示词，不要解释。"""

# 构建图片信息
image_info_lines = [f"- 图片{image_index_map.get('keyframe', 1)}：关键帧/场景参考"]
product_idx = image_index_map.get('product')
if product_idx:
    image_info_lines.append(f"- 图片{product_idx}：产品图（K7主机）")

character_desc_lines = []
for c in test_characters:
    char_id = c.get("id", "")
    char_desc = c.get("description", "无描述")
    selected_images = c.get("selected_images", ["front_full"])
    
    img_refs = []
    for img_type in selected_images:
        key = f"{char_id}_{img_type}"
        img_idx = image_index_map.get(key)
        if img_idx:
            img_refs.append(f"图片{img_idx}")
    
    img_ref_str = "、".join(img_refs) if img_refs else "未分配图片"
    character_desc_lines.append(f"- {char_id}：{char_desc}（对应{img_ref_str}）")
    image_info_lines.append(f"- {img_ref_str}：人物【{char_id}】的{'+'.join(selected_images)}图")

character_desc = "\n".join(character_desc_lines)
image_index_info = "\n".join(image_info_lines)

user_prompt = f"""## 图片编号映射（必读！）
{image_index_info}

## 任务信息
- 时长：5秒
- 人物数量：{len(test_characters)}
- 人物详情：
{character_desc}

## 原始提示词
{test_initial_prompt}

## 脚本信息
{json.dumps(test_segment, ensure_ascii=False, indent=2)}

请根据以上信息，特别是【图片编号映射】和【人物数量】，优化这个提示词。关键要求：
1. 明确标注每个人物对应哪张图片
2. 人物名称前后一致
3. 使用正确的图片编号引用"""

print("="*60)
print("发送给 GPT 的 System Prompt (前1000字符):")
print("="*60)
print(system_prompt[:1000])
print(f"\n... (共 {len(system_prompt)} 字符)")

print("\n" + "="*60)
print("发送给 GPT 的 User Prompt:")
print("="*60)
print(user_prompt)